In [ ]:
%pip install kagglehub tensorflow pandas numpy scikit-learn pyserial paho-mqtt

^C
Note: you may need to restart the kernel to use updated packages.


In [3]:
import kagglehub

path = kagglehub.dataset_download("nvnikhil0001/sis-fall-original-dataset")
print(path)

c:\Users\ASUS GAMING\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\ASUS GAMING\.cache\kagglehub\datasets\nvnikhil0001\sis-fall-original-dataset\versions\1


In [4]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [6]:
def read_file(filepath):
    data = []
    with open(filepath) as f:
        for line in f:
            parts = line.strip().replace(';', ',').split(',')

            if len(parts) < 6:
                parts = line.strip().split()

            if len(parts) < 6:
                continue

            try:
                ax, ay, az, gx, gy, gz = map(float, parts[:6])
                data.append([ax, ay, az, gx, gy, gz])
            except:
                continue

    return data

In [7]:
all_data = []

for root, dirs, files in os.walk(path):
    for file in files:
        if not file.endswith(".txt"):
            continue

        label = 1 if file.startswith("F") else 0 if file.startswith("D") else None
        if label is None:
            continue

        file_path = os.path.join(root, file)
        data = read_file(file_path)

        for row in data:
            ax, ay, az, gx, gy, gz = row

            # 🔥 TÍNH ROLL / PITCH
            roll = np.arctan2(ay, az)
            pitch = np.arctan2(-ax, np.sqrt(ay**2 + az**2))

            # 🔥 FAKE SENSOR (để match ESP32 input)
            bpm = 70 + np.random.normal(0, 3)
            spo2 = 97 + np.random.normal(0, 1)
            temp = 36.5 + np.random.normal(0, 0.2)

            all_data.append([
                ax, ay, az,
                gx, gy, gz,
                roll, pitch,
                bpm, spo2, temp,
                label
            ])

df = pd.DataFrame(all_data, columns=[
    'ax','ay','az','gx','gy','gz',
    'roll','pitch',
    'bpm','spo2','temp',
    'label'
])

print(df.shape)

(15858929, 12)


In [8]:
WINDOW_SIZE = 150
STRIDE = 50

def create_windows(df):
    windows = []
    labels = []

    cols = df.columns[:-1]
    data = df[cols].values
    y = df['label'].values

    for i in range(0, len(data) - WINDOW_SIZE, STRIDE):
        window = data[i:i+WINDOW_SIZE]
        segment = y[i:i+WINDOW_SIZE]

        label = 1 if np.max(segment) == 1 else 0

        windows.append(window)
        labels.append(label)

    return np.array(windows), np.array(labels)

windows, labels = create_windows(df)

print("Windows:", windows.shape)

Windows: (317176, 150, 11)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    windows, labels, test_size=0.2, random_state=42
)

In [10]:
mean = np.mean(X_train, axis=(0,1))
std = np.std(X_train, axis=(0,1)) + 1e-6

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

In [11]:
model = tf.keras.Sequential([

    # CNN
    tf.keras.layers.Conv1D(64, 3, activation='relu', input_shape=(150,11)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling1D(2),

    tf.keras.layers.Conv1D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling1D(2),

    # 🔥 LSTM
    tf.keras.layers.LSTM(64, return_sequences=False),

    tf.keras.layers.Dropout(0.4),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

c:\Users\ASUS GAMING\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [13]:
history = model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/25
7930/7930 ━━━━━━━━━━━━━━━━━━━━ 110s 13ms/step - accuracy: 0.8205 - loss: 0.4144 - val_accuracy: 0.8579 - val_loss: 0.3409
Epoch 2/25
7930/7930 ━━━━━━━━━━━━━━━━━━━━ 98s 12ms/step - accuracy: 0.8609 - loss: 0.3360 - val_accuracy: 0.8747 - val_loss: 0.3013
Epoch 3/25
7930/7930 ━━━━━━━━━━━━━━━━━━━━ 99s 12ms/step - accuracy: 0.8756 - loss: 0.3033 - val_accuracy: 0.8863 - val_loss: 0.2758
Epoch 4/25
7930/7930 ━━━━━━━━━━━━━━━━━━━━ 99s 13ms/step - accuracy: 0.8833 - loss: 0.2844 - val_accuracy: 0.8952 - val_loss: 0.2584
Epoch 5/25
7930/7930 ━━━━━━━━━━━━━━━━━━━━ 100s 13ms/step - accuracy: 0.8883 - loss: 0.2714 - val_accuracy: 0.8980 - val_loss: 0.2486
Epoch 6/25
7930/7930 ━━━━━━━━━━━━━━━━━━━━ 99s 13ms/step - accuracy: 0.8924 - loss: 0.2626 - val_accuracy: 0.8972 - val_loss: 0.2539
Epoch 7/25
7930/7930 ━━━━━━━━━━━━━━━━━━━━ 100s 13ms/step - accuracy: 0.8955 - loss: 0.2548 - val_accuracy: 0.9006 - val_loss: 0.2444
Epoch 8/25
7930/7930 ━━━━━━━━━━━━━━━━━━━━ 99s 12ms/step - accuracy: 0.898

In [14]:
loss, acc = model.evaluate(X_test, y_test)
print("Accuracy:", acc)

1983/1983 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9179 - loss: 0.2016
Accuracy: 0.9179330468177795


In [15]:
y_pred = (model.predict(X_test) > 0.6).astype(int)

print(classification_report(y_test, y_pred))

1983/1983 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step
              precision    recall  f1-score   support

           0       0.91      0.97      0.94     42150
           1       0.93      0.81      0.86     21286

    accuracy                           0.91     63436
   macro avg       0.92      0.89      0.90     63436
weighted avg       0.92      0.91      0.91     63436



In [21]:
def predict_from_json(data, model, mean, std):
    sample = np.array([
        data['ax'], data['ay'], data['az'],
        data['gx'], data['gy'], data['gz'],
        data['roll'], data['pitch'],
        data['bpm'], data['spo2'], data['temp']
    ])

    # tạo window giả (lặp lại)
    window = np.tile(sample, (150,1))

    window = (window - mean) / std
    window = window.reshape(1,150,11)

    prob = model.predict(window)[0][0]

    return prob

# test
data = {
  "bpm":72,"spo2":98,"temp":36.5,
  "ax":0.01,"ay":0.02,"az":1.0,
  "gx":0.1,"gy":-0.2,"gz":0.0,
  "roll":5.2,"pitch":-1.3
}

print("Fall probability:", predict_from_json(data, model, mean, std))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
Fall probability: 0.99857855


In [18]:
# Save trained Keras model to workspace root
from pathlib import Path

BASE_DIR = Path(r"d:\Ky6\PBL")
BASE_DIR.mkdir(parents=True, exist_ok=True)

out_h5 = BASE_DIR / "model.h5"
out_dir = BASE_DIR / "model_saved"

try:
    model.save(str(out_h5))
    print(f"Saved Keras model to {out_h5}")
except Exception as e:
    print(f"Could not save to {out_h5}: {e}\nFalling back to SavedModel directory: {out_dir}")
    model.save(str(out_dir))
    print(f"Saved Keras model to {out_dir}")

Saved Keras model to d:\Ky6\PBL\model.h5


In [26]:
# Save normalization stats for realtime inference
from pathlib import Path
import numpy as np

BASE_DIR = Path(r"d:\Ky6\PBL")
BASE_DIR.mkdir(parents=True, exist_ok=True)

np.save(BASE_DIR / "mean.npy", mean)
np.save(BASE_DIR / "std.npy", std)

print(f"Saved: {BASE_DIR / 'mean.npy'}")
print(f"Saved: {BASE_DIR / 'std.npy'}")

Saved: d:\Ky6\PBL\mean.npy
Saved: d:\Ky6\PBL\std.npy


In [27]:
from pathlib import Path
from collections import deque
import json
import numpy as np
import tensorflow as tf

# ===== Realtime inference config =====
BASE_DIR = Path(r"d:\Ky6\PBL")
MODEL_H5 = BASE_DIR / "model.h5"
MODEL_SAVED = BASE_DIR / "model_saved"
MEAN_FILE = BASE_DIR / "mean.npy"
STD_FILE = BASE_DIR / "std.npy"
JSON_STREAM_FILE = BASE_DIR / "sensor_stream.jsonl"

WINDOW_SIZE = 150
THRESHOLD = 0.6
FEATURE_KEYS = ["ax", "ay", "az", "gx", "gy", "gz", "roll", "pitch", "bpm", "spo2", "temp"]

# ===== Load model =====
if MODEL_H5.exists():
    prod_model = tf.keras.models.load_model(str(MODEL_H5))
    print(f"Loaded model: {MODEL_H5}")
elif MODEL_SAVED.exists():
    prod_model = tf.keras.models.load_model(str(MODEL_SAVED))
    print(f"Loaded model: {MODEL_SAVED}")
else:
    raise FileNotFoundError(f"Model not found in {BASE_DIR}")

# ===== Load normalization stats =====
if "mean" in globals() and "std" in globals():
    prod_mean = np.array(mean, dtype=np.float32)
    prod_std = np.array(std, dtype=np.float32)
    print("Using mean/std from current notebook session")
elif MEAN_FILE.exists() and STD_FILE.exists():
    prod_mean = np.load(MEAN_FILE)
    prod_std = np.load(STD_FILE)
    print(f"Loaded normalization stats: {MEAN_FILE}, {STD_FILE}")
else:
    raise FileNotFoundError(
        "mean/std not found. Save them after training: np.save('d:/Ky6/PBL/mean.npy', mean); np.save('d:/Ky6/PBL/std.npy', std)"
    )

# ===== Handling invalid/missing physiological values =====
DEFAULTS = {
    "bpm": 72.0,
    "spo2": 97.0,
    "temp": 36.5,
}
last_valid = DEFAULTS.copy()


def to_float(value, fallback):
    try:
        return float(value)
    except (TypeError, ValueError):
        return float(fallback)


def enrich_sample(sample: dict) -> dict:
    out = dict(sample)

    # Required for this model architecture.
    required_motion = ["ax", "ay", "az", "gx", "gy", "gz", "temp"]
    for key in required_motion:
        if key not in out:
            raise KeyError(f"Missing key '{key}'")

    # Force numeric conversion.
    out["ax"] = to_float(out["ax"], 0.0)
    out["ay"] = to_float(out["ay"], 0.0)
    out["az"] = to_float(out["az"], 1.0)
    out["gx"] = to_float(out["gx"], 0.0)
    out["gy"] = to_float(out["gy"], 0.0)
    out["gz"] = to_float(out["gz"], 0.0)
    out["temp"] = to_float(out["temp"], last_valid["temp"])

    # Arduino does not need to send roll/pitch; compute on backend if absent.
    if "roll" not in out or "pitch" not in out:
        ax, ay, az = out["ax"], out["ay"], out["az"]
        out["roll"] = float(np.arctan2(ay, az))
        out["pitch"] = float(np.arctan2(-ax, np.sqrt(ay ** 2 + az ** 2)))
    else:
        out["roll"] = to_float(out["roll"], 0.0)
        out["pitch"] = to_float(out["pitch"], 0.0)

    # Replace invalid bpm/spo2 with last valid values.
    bpm_raw = to_float(out.get("bpm", -1), -1)
    if bpm_raw > 0:
        last_valid["bpm"] = bpm_raw
    out["bpm"] = last_valid["bpm"]

    spo2_raw = to_float(out.get("spo2", -1), -1)
    if 50 <= spo2_raw <= 100:
        last_valid["spo2"] = spo2_raw
    out["spo2"] = last_valid["spo2"]

    # Keep only realistic body temperature; otherwise reuse last valid value.
    if 25 <= out["temp"] <= 45:
        last_valid["temp"] = out["temp"]
    out["temp"] = last_valid["temp"]

    for key in FEATURE_KEYS:
        if key not in out:
            raise KeyError(f"Missing key '{key}' after preprocessing")

    return out


def sample_to_vector(sample: dict) -> np.ndarray:
    return np.array([float(sample[k]) for k in FEATURE_KEYS], dtype=np.float32)


def iter_json_samples(jsonl_path: Path):
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                payload = json.loads(line)
                yield line_no, payload
            except Exception as e:
                print(f"Skip line {line_no}: invalid JSON ({e})")


# ===== Quick check with your real payload format =====
real_payload_example = {
    "ts": 24190,
    "bpm": -1,
    "spo2": -1,
    "temp": 33.71,
    "ax": 0.9155,
    "ay": -0.1921,
    "az": -0.1262,
    "gx": -1.2828,
    "gy": 17.5719,
    "gz": 1.2565,
    "ir": 13335,
    "red": 7680,
}

_check = enrich_sample(real_payload_example)
print(f"Example accepted. Computed roll={_check['roll']:.4f}, pitch={_check['pitch']:.4f}")

# ===== Streaming inference with 150-sample window =====
if not JSON_STREAM_FILE.exists():
    raise FileNotFoundError(f"JSON stream file not found: {JSON_STREAM_FILE}")

buffer = deque(maxlen=WINDOW_SIZE)
for line_no, raw_sample in iter_json_samples(JSON_STREAM_FILE):
    try:
        sample = enrich_sample(raw_sample)
        vec = sample_to_vector(sample)
        buffer.append(vec)
    except Exception as e:
        print(f"Skip line {line_no}: {e}")
        continue

    if len(buffer) < WINDOW_SIZE:
        continue

    window = np.array(buffer, dtype=np.float32)
    window = (window - prod_mean) / (prod_std + 1e-6)
    window = window.reshape(1, WINDOW_SIZE, len(FEATURE_KEYS))

    prob = float(prod_model.predict(window, verbose=0)[0][0])
    pred = int(prob > THRESHOLD)
    status = "FALL_ALERT" if pred == 1 else "NORMAL"
    print(f"line={line_no} | prob={prob:.4f} | pred={pred} | {status}")

Loaded model: d:\Ky6\PBL\model.h5
Using mean/std from current notebook session
Example accepted. Computed roll=-2.1520, pitch=-1.3248
line=150 | prob=0.9751 | pred=1 | FALL_ALERT
line=151 | prob=0.9881 | pred=1 | FALL_ALERT
line=152 | prob=0.9830 | pred=1 | FALL_ALERT
line=153 | prob=0.9928 | pred=1 | FALL_ALERT
line=154 | prob=0.9938 | pred=1 | FALL_ALERT
line=155 | prob=0.9948 | pred=1 | FALL_ALERT
line=156 | prob=0.9930 | pred=1 | FALL_ALERT
line=157 | prob=0.9961 | pred=1 | FALL_ALERT
line=158 | prob=0.9964 | pred=1 | FALL_ALERT
line=159 | prob=0.9964 | pred=1 | FALL_ALERT
line=160 | prob=0.9952 | pred=1 | FALL_ALERT
line=161 | prob=0.9978 | pred=1 | FALL_ALERT
line=162 | prob=0.9979 | pred=1 | FALL_ALERT
line=163 | prob=0.9980 | pred=1 | FALL_ALERT
line=164 | prob=0.9973 | pred=1 | FALL_ALERT
line=165 | prob=0.9983 | pred=1 | FALL_ALERT
line=166 | prob=0.9984 | pred=1 | FALL_ALERT
line=167 | prob=0.9985 | pred=1 | FALL_ALERT
line=168 | prob=0.9980 | pred=1 | FALL_ALERT
line=169 | 

In [ ]:
# Optional: realtime inference directly from Arduino Serial
# Install once if needed: %pip install pyserial
import json
import sys
import subprocess
from collections import deque

import numpy as np

try:
    import serial
    from serial import SerialException
    from serial.tools import list_ports
except ModuleNotFoundError:
    print("Installing pyserial...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyserial"])
    import serial
    from serial import SerialException
    from serial.tools import list_ports

# This cell depends on objects created in Cell 17.
required_objects = [
    "WINDOW_SIZE",
    "FEATURE_KEYS",
    "prod_model",
    "prod_mean",
    "prod_std",
    "enrich_sample",
    "sample_to_vector",
]
missing = [name for name in required_objects if name not in globals()]
if missing:
    raise RuntimeError(f"Run Cell 17 first. Missing: {missing}")

BAUD = 115200
FORCE_PORT = None  # Set e.g. "COM9" to force one port

ports = list(list_ports.comports())
if not ports:
    raise RuntimeError("No serial port found. Connect ESP32 and retry.")

print("Available ports:")
for p in ports:
    print(f" - {p.device}: {p.description}")

if FORCE_PORT:
    candidates = [FORCE_PORT]
else:
    preferred = [
        p.device
        for p in ports
        if (
            "USB" in p.description.upper()
            or "CH340" in p.description.upper()
            or "CP210" in p.description.upper()
            or "UART" in p.description.upper()
        )
    ]
    others = [p.device for p in ports if p.device not in preferred]
    candidates = preferred + others

ser = None
last_error = None
for port_name in candidates:
    try:
        ser = serial.Serial(port_name, BAUD, timeout=1)
        PORT = port_name
        break
    except SerialException as e:
        last_error = e
        print(f"Cannot open {port_name}: {e}")

if ser is None:
    raise RuntimeError(
        "Could not open any COM port. Close Arduino IDE Serial Monitor/Plotter and any app using COM, then rerun Cell 18. "
        "If you only need MQTT input, run Cell 19 instead."
    ) from last_error

print(f"Listening on {PORT} at {BAUD}... Stop the cell to end.")

serial_buffer = deque(maxlen=WINDOW_SIZE)

try:
    while True:
        line = ser.readline().decode("utf-8", errors="ignore").strip()
        if not line:
            continue

        # Accept raw JSON or prefixed logs, e.g. "[MQTT] Published -> {...}".
        start = line.find("{")
        end = line.rfind("}")
        if start == -1 or end == -1 or end <= start:
            continue

        payload_text = line[start : end + 1]
        try:
            raw = json.loads(payload_text)
        except Exception:
            continue

        try:
            sample = enrich_sample(raw)
            serial_buffer.append(sample_to_vector(sample))
        except Exception as e:
            print(f"Skip serial sample: {e}")
            continue

        if len(serial_buffer) < WINDOW_SIZE:
            print(f"Warmup {len(serial_buffer)}/{WINDOW_SIZE}")
            continue

        window = np.array(serial_buffer, dtype=np.float32)
        window = (window - prod_mean) / (prod_std + 1e-6)
        window = window.reshape(1, WINDOW_SIZE, len(FEATURE_KEYS))

        prob = float(prod_model.predict(window, verbose=0)[0][0])
        pred = int(prob > THRESHOLD)
        status = "FALL_ALERT" if pred == 1 else "NORMAL"
        print(f"prob={prob:.4f} | pred={pred} | {status}")
except KeyboardInterrupt:
    print("Stopped serial inference.")
finally:
    ser.close()

Available ports:
 - COM4: Standard Serial over Bluetooth link (COM4)
 - COM9: Silicon Labs CP210x USB to UART Bridge (COM9)
 - COM3: Standard Serial over Bluetooth link (COM3)
 - COM5: Standard Serial over Bluetooth link (COM5)
 - COM7: Standard Serial over Bluetooth link (COM7)
 - COM6: Standard Serial over Bluetooth link (COM6)
 - COM8: Standard Serial over Bluetooth link (COM8)
Using port: COM9


SerialException: could not open port 'COM9': PermissionError(13, 'Access is denied.', None, 5)

In [ ]:
# MQTT -> model inference -> local file output (no MQTT publish)
import json
import sys
import time
import subprocess
from collections import defaultdict, deque
from pathlib import Path

import numpy as np

try:
    import paho.mqtt.client as mqtt
except ModuleNotFoundError:
    print("Installing paho-mqtt...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "paho-mqtt"])
    import paho.mqtt.client as mqtt

# This cell depends on objects created in Cell 17.
required_objects = [
    "BASE_DIR",
    "WINDOW_SIZE",
    "FEATURE_KEYS",
    "THRESHOLD",
    "prod_model",
    "prod_mean",
    "prod_std",
    "enrich_sample",
    "sample_to_vector",
]
missing = [name for name in required_objects if name not in globals()]
if missing:
    raise RuntimeError(f"Run Cell 17 first. Missing: {missing}")

# ===== MQTT source (match ESP32 firmware) =====
BROKER = "11060dbd13b54fc988ae8f9bfc43c089.s1.eu.hivemq.cloud"
PORT = 8883
MQTT_USER = "heart-rate"
MQTT_PASS = "aB123456"
TOPIC_IN = "test"
CLIENT_ID = f"fall-model-file-writer-{int(time.time())}"

# ===== Local output file =====
OUTPUT_FILE = Path(BASE_DIR) / "mqtt_model_predictions.jsonl"
RESET_OUTPUT_FILE = False  # set True to clear file on each run

if RESET_OUTPUT_FILE and OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()

print(f"Reading MQTT topic: {TOPIC_IN}")
print(f"Writing predictions to: {OUTPUT_FILE}")

buffers = defaultdict(lambda: deque(maxlen=WINDOW_SIZE))


def append_jsonl(path: Path, obj: dict) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("Connected to HiveMQ Cloud")
        client.subscribe(TOPIC_IN)
        print(f"Subscribed: {TOPIC_IN}")
    else:
        print(f"MQTT connect failed with rc={rc}")


def on_message(client, userdata, msg):
    try:
        raw = json.loads(msg.payload.decode("utf-8"))
    except Exception as e:
        print(f"Invalid JSON: {e}")
        return

    device_id = str(raw.get("device_id", raw.get("device", "esp32")))

    try:
        sample = enrich_sample(raw)
        vec = sample_to_vector(sample)
    except Exception as e:
        print(f"Skip payload from {device_id}: {e}")
        return

    buf = buffers[device_id]
    buf.append(vec)

    if len(buf) < WINDOW_SIZE:
        if len(buf) in (1, 25, 50, 75, 100, 125, 149):
            print(f"[{device_id}] warmup {len(buf)}/{WINDOW_SIZE}")
        return

    window = np.array(buf, dtype=np.float32)
    window = (window - prod_mean) / (prod_std + 1e-6)
    window = window.reshape(1, WINDOW_SIZE, len(FEATURE_KEYS))

    prob = float(prod_model.predict(window, verbose=0)[0][0])
    pred = int(prob > THRESHOLD)
    status = "FALL_ALERT" if pred == 1 else "NORMAL"

    result = {
        "server_ts": int(time.time() * 1000),
        "device_id": device_id,
        "status": status,
        "pred": pred,
        "fall_probability": round(prob, 4),
        "threshold": THRESHOLD,
        "features": {
            "bpm": float(sample["bpm"]),
            "spo2": float(sample["spo2"]),
            "temp": float(sample["temp"]),
            "ax": float(sample["ax"]),
            "ay": float(sample["ay"]),
            "az": float(sample["az"]),
            "gx": float(sample["gx"]),
            "gy": float(sample["gy"]),
            "gz": float(sample["gz"]),
            "roll": float(sample["roll"]),
            "pitch": float(sample["pitch"]),
        },
        "raw": raw,
    }

    append_jsonl(OUTPUT_FILE, result)
    print(f"[{device_id}] prob={prob:.4f} pred={pred} status={status} -> saved")


mqtt_client = mqtt.Client(client_id=CLIENT_ID, clean_session=True, protocol=mqtt.MQTTv311)
mqtt_client.username_pw_set(MQTT_USER, MQTT_PASS)
mqtt_client.tls_set()

mqtt_client.on_connect = on_connect
mqtt_client.on_message = on_message

mqtt_client.connect(BROKER, PORT, keepalive=60)
print("Collector running... stop this cell to end.")
mqtt_client.loop_forever()

ModuleNotFoundError: No module named 'paho'

In [ ]:
# View latest predictions written from MQTT stream
from pathlib import Path
import json

N = 10
base_dir = Path(BASE_DIR) if "BASE_DIR" in globals() else Path(r"d:\Ky6\PBL")
output_file = base_dir / "mqtt_model_predictions.jsonl"

if not output_file.exists():
    print(f"Output file not found: {output_file}")
else:
    with open(output_file, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    print(f"Total predictions: {len(lines)}")
    print(f"Showing last {min(N, len(lines))} records:")

    for line in lines[-N:]:
        rec = json.loads(line)
        brief = {
            "server_ts": rec.get("server_ts"),
            "device_id": rec.get("device_id"),
            "status": rec.get("status"),
            "pred": rec.get("pred"),
            "fall_probability": rec.get("fall_probability"),
            "ts": rec.get("ts", rec.get("raw", {}).get("ts")),
        }
        print(json.dumps(brief, ensure_ascii=False))